# Tune SEI conductivities (kappa_EC, kappa_VC)

**Context**: This notebook sweeps the SEI **conductivities** (`kappa_EC` and
`kappa_VC`) while holding the SEI **diffusivities** (`D_LEDC`, `D_LVDC`) and
**densities** (`rho_EC`, `rho_VC`) fixed at their previously-tuned values.
This corresponds to the "Set conductivities" stage of the SEI parameter
tuning workflow.

This is a copy of just the **conductivities** portion of
`2026_02_25_ec_tuning.ipynb`, extracted into its own notebook so it can be
re-run independently of the diffusivity and density tuning steps.

The objective function is the absolute error between the simulated SEI
resistance (`R_sei + 0.01`) and the measured pre-aging resistance from the
HPPC pulse, evaluated at the time of that pulse for each formation protocol
(base / fast / fast+).

Pipeline:

1. Sweep `kappa_EC` × `kappa_VC` over a 35×36 grid using fixed diffusivities
   and tuned densities, with `include_flag=2` (formation + initial HPPC).
2. Compute the resistance error against the measured HPPC values.
3. Pickle the resulting error matrices.
4. Plot resistance-error heatmaps via `simutils.plot_heatmaps_kappa`.

Andrew Weng

In [1]:
import os, sys

if os.path.basename(os.getcwd()) in ('notebooks', 'notebooks_2026'):
    os.chdir('../')
    sys.path.insert(0, 'src/')

from src import cellsim as cellsim

from scipy.interpolate import interp1d

import pickle
import numpy as np
from matplotlib import pyplot as plt
from src import plotter as plotter
from src import ioutils as ioutils
from src import simutils as simutils

%load_ext autoreload
%matplotlib inline

plotter.initialize(plt)

# Set conductivities

In [ ]:

rs_meas1_base = 21.5e-3
rs_meas1_fast = 17.7e-3
rs_meas1_fast_plus = 16.6e-3

t_hppc1_base = 69.39*3600
t_hppc1_fast = 16.17*3600
t_hppc1_fast_plus = 7.465*3600
rs_meas1_time_base = t_hppc1_base + 54000
rs_meas1_time_fast = t_hppc1_fast + 54000
rs_meas1_time_fast_plus = t_hppc1_fast_plus + 54000
rs_meas2_time = 505.23 * 3600 + 5e4
rs_meas2_base = 0.0375
rs_meas2_fast = 0.0366
rs_meas2_fast_plus = 0.0366

nn = 35

kappa_ec_vec = np.linspace(5e-5, 5e-4, nn)
kappa_vc_vec = np.linspace(1e-5, 1e-4, nn+1)

err1_mat = np.zeros((len(kappa_ec_vec), len(kappa_vc_vec)))
err2_mat = np.zeros((len(kappa_ec_vec), len(kappa_vc_vec)))
err3_mat = np.zeros((len(kappa_ec_vec), len(kappa_vc_vec)))
count = 0

# Tuning based on voltage
d_ec_opt1 = 6.66e-20
d_ec_opt2 = 6.66e-20
d_ec_opt3 = 6.66e-20
d_vc_opt1 = 1.22e-18
d_vc_opt2 = 1.22e-18
d_vc_opt3 = 1.22e-18

# Optimized densities from baseline formation, expansion RMSE
rho_vc_opt = 1.5
rho_ec_opt1 = 1.94
rho_ec_opt2 = 2.25
rho_ec_opt3 = 1.63

for i, kappa_ec in enumerate(kappa_ec_vec):
    for j, kappa_vc in enumerate(kappa_vc_vec):

        print(f'Running kappa_EC = {kappa_ec:.2e} m2/s and kappa_VC = {kappa_vc:.2e} ({count} of {nn*(nn+1)})')

        try:
            df_sim1 = simutils.run_sim('base',  diff_ec=d_ec_opt1, diff_vc=d_vc_opt1, kappa_ec=kappa_ec, kappa_vc=kappa_vc, rho_ec=rho_ec_opt1, rho_vc=rho_vc_opt, dt=1, include_flag=2)
            df_sim2 = simutils.run_sim('fast',  diff_ec=d_ec_opt2, diff_vc=d_vc_opt2, kappa_ec=kappa_ec, kappa_vc=kappa_vc, rho_ec=rho_ec_opt2, rho_vc=rho_vc_opt, dt=1, include_flag=2)
            df_sim3 = simutils.run_sim('fast+', diff_ec=d_ec_opt3, diff_vc=d_vc_opt3, kappa_ec=kappa_ec, kappa_vc=kappa_vc, rho_ec=rho_ec_opt3, rho_vc=rho_vc_opt, dt=1, include_flag=2)
        except:
            print('ERROR')
            continue

        interp_rsei_base = interp1d(df_sim1['t'], df_sim1['R_sei'] + 0.01, bounds_error=False, fill_value="extrapolate")
        rs_modl1 = interp_rsei_base(rs_meas1_time_base)
        err1 = np.abs(rs_modl1 - rs_meas1_base)

        interp_rsei_fast = interp1d(df_sim2['t'], df_sim2['R_sei'] + 0.01, bounds_error=False, fill_value="extrapolate")
        rs_modl2 = interp_rsei_fast(rs_meas1_time_fast)
        err2 = np.abs(rs_modl2 - rs_meas1_fast)

        interp_rsei_fast_plus = interp1d(df_sim3['t'], df_sim3['R_sei'] + 0.01, bounds_error=False, fill_value="extrapolate")
        rs_modl3 = interp_rsei_fast_plus(rs_meas1_time_fast_plus)
        err3 = np.abs(rs_modl3 - rs_meas1_fast_plus)

        err1_mat[i, j] = err1
        err2_mat[i, j] = err2
        err3_mat[i, j] = err3

        count += 1


# Replace zeros with np.nan (failed runs)
rmse1_e_mat = np.where(err1_mat == 0, np.nan, err1_mat)
rmse2_e_mat = np.where(err2_mat == 0, np.nan, err2_mat)
rmse3_e_mat = np.where(err3_mat == 0, np.nan, err3_mat)

Running kappa_EC = 5.00e-05 m2/s and kappa_VC = 1.00e-05 (0 of 1260)
Running kappa_EC = 5.00e-05 m2/s and kappa_VC = 1.26e-05 (1 of 1260)
Running kappa_EC = 5.00e-05 m2/s and kappa_VC = 1.51e-05 (2 of 1260)


In [ ]:
variables_to_pickle = {
    'rmse1_e_mat': rmse1_e_mat,
    'rmse2_e_mat': rmse2_e_mat,
    'rmse3_e_mat': rmse3_e_mat,
}

with open('notebooks_2026/data/resistance_optimization_results_20260504.pkl', 'wb') as f:
    pickle.dump(variables_to_pickle, f)

# Visualize results

In [ ]:
%autoreload 2
%matplotlib inline

# Load the pickled simulation results
with open('notebooks_2026/data/resistance_optimization_results_20260504.pkl', 'rb') as f:
    sim_results = pickle.load(f)

nn = 35
kappa_ec_vec = np.linspace(5e-5, 5e-4, nn)
kappa_vc_vec = np.linspace(1e-5, 1e-4, nn+1)

# Extract variables from the loaded results (convert to milli-units for display)
er1_mat = sim_results['rmse1_e_mat']*1e3
er2_mat = sim_results['rmse2_e_mat']*1e3
er3_mat = sim_results['rmse3_e_mat']*1e3

simutils.plot_heatmaps_kappa(
    kappa_ec_vec*1e3, kappa_vc_vec*1e3,
    er1_mat, er2_mat, er3_mat,
    r'|$R - \hat{R}|$ (m$\Omega$)',
    zmin=0.000, zmax=10,
    to_annotate=True,
    annotation_orientation='vertical',
    annotation_coord=0.05,
    tosave=True,
    savename='outputs/figures/heatmap_resistance_error.svg',
)

In [ ]:
kappa_vc_opt1 = 0.0511e-3
kappa_vc_opt2 = 0.0511e-3
kappa_vc_opt3 = 0.0511e-3
kappa_ec_opt1 = 0.0765e-3
kappa_ec_opt2 = 0.0897e-3
kappa_ec_opt3 = 0.1290e-3
